# Reinforcement Learning: MDP Foundations and Policy Gradient

**Sequential decision-making** is the core problem: an agent makes a sequence of decisions, each affecting future states, with the goal of maximizing cumulative reward. Unlike supervised learning, there are no labels — only scalar reward signals that evaluate whether a trajectory was good or bad.

Two compounding challenges:
1. **Long-term consequences** — the greedy action now may be suboptimal later
2. **No direct supervision** — reward tells you outcome quality, not which action was optimal

This notebook builds the full mathematical stack from scratch:
- Markov Decision Process (MDP) formalism
- Policies: deterministic and stochastic
- Value functions and Bellman equations
- Policy Gradient (REINFORCE) derivation and implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12
})
print('Libraries loaded.')

---
## 1. Markov Decision Process (MDP)

An MDP is a 5-tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

| Symbol | Meaning |
|--------|--------|
| $\mathcal{S}$ | Set of states — all possible configurations of the world |
| $\mathcal{A}$ | Set of actions — all possible decisions the agent can take |
| $P_{sa}(s')$ | Transition dynamics — probability of reaching $s'$ from state $s$ after action $a$ |
| $R : \mathcal{S} \to \mathbb{R}$ | Reward function — scalar evaluation of each state |
| $\gamma \in [0, 1)$ | Discount factor — how much to weight future rewards |

### Markov Property

The transition distribution depends only on the current state and action, not on history:

$$P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots) = P(s_{t+1} \mid s_t, a_t) = P_{s_t, a_t}(s_{t+1})$$

This is what makes the problem tractable — the past is fully summarized by the current state.

### Interaction loop

$$s_0 \xrightarrow{a_0} s_1 \xrightarrow{a_1} s_2 \xrightarrow{a_2} \cdots$$

where $s_{t+1} \sim P_{s_t, a_t}$ and $a_t$ is chosen by the agent's policy.

In [ ]:
class GridMDP:
    """1D grid world MDP with stochastic transitions."""

    def __init__(self, n_states=10, goal=9, gamma=0.99, slip_prob=0.1):
        self.n_states = n_states
        self.goal = goal
        self.gamma = gamma
        self.slip_prob = slip_prob          # probability of going opposite direction
        self.actions = ['left', 'right']    # action set A
        self.P = self._build_transitions()
        self.R = self._build_rewards()

    def _build_transitions(self):
        # P[s][a] = probability vector over next states (shape: n_states)
        P = {}
        for s in range(self.n_states):
            P[s] = {}
            for a_idx, a in enumerate(self.actions):
                probs = np.zeros(self.n_states)
                intended = s - 1 if a == 'left' else s + 1
                slipped  = s + 1 if a == 'left' else s - 1
                intended = np.clip(intended, 0, self.n_states - 1)
                slipped  = np.clip(slipped,  0, self.n_states - 1)
                probs[intended] += (1 - self.slip_prob)
                probs[slipped]  += self.slip_prob
                P[s][a] = probs
        return P

    def _build_rewards(self):
        R = np.full(self.n_states, -0.1)    # small penalty for non-goal states
        R[self.goal] = 1.0                   # positive reward at goal
        return R

    def step(self, s, a):
        probs = self.P[s][a]
        s_next = np.random.choice(self.n_states, p=probs)
        return s_next, self.R[s_next]


mdp = GridMDP(n_states=10, goal=9, gamma=0.99, slip_prob=0.1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Visualize reward function R(s)
axes[0].bar(range(mdp.n_states), mdp.R, color=['#2ecc71' if s == mdp.goal else '#e74c3c'
                                                for s in range(mdp.n_states)])
axes[0].set_xlabel('State s')
axes[0].set_ylabel('R(s)')
axes[0].set_title('Reward Function R(s)')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

# Visualize transition distribution P(s=5, a='right')
s_demo, a_demo = 5, 'right'
axes[1].bar(range(mdp.n_states), mdp.P[s_demo][a_demo], color='steelblue')
axes[1].set_xlabel('Next state s\'')
axes[1].set_ylabel('Probability')
axes[1].set_title(f'Transition distribution $P_{{s={s_demo}, a={a_demo}}}(s\')$')

plt.tight_layout()
plt.savefig('/tmp/mdp_overview.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'States: {list(range(mdp.n_states))}  |  Actions: {mdp.actions}  |  γ={mdp.gamma}')

---
## 2. Discounted Return and Bounded Payoff

For a trajectory $\tau = (s_0, a_0, s_1, a_1, \ldots, s_T)$, the **discounted return** is:

$$G(\tau) = \sum_{t=0}^{T} \gamma^t R(s_t)$$

The discount factor $\gamma \in [0,1)$ serves two purposes:

1. **Economic interpretation** — reward earned sooner is worth more (like interest rates)
2. **Mathematical convenience** — even for infinite horizons, the return is bounded

### Bound on infinite-horizon return

If $|R(s)| \leq M$ for all $s$, then using the geometric series formula:

$$|G(\tau)| \leq \sum_{t=0}^{\infty} \gamma^t M = M \sum_{t=0}^{\infty} \gamma^t = \frac{M}{1-\gamma}$$

This allows us to speak meaningfully about infinite-horizon problems — the expected return is always finite.

### Why discount encourages speed

Getting reward $+1$ at time step $t$ contributes $\gamma^t$ to the return. With $\gamma = 0.99$:

- Step 1: contributes $0.99$
- Step 50: contributes $0.99^{50} \approx 0.605$
- Step 100: contributes $0.99^{100} \approx 0.366$

The agent is incentivized to reach the goal as quickly as possible.

In [ ]:
def discounted_return(rewards, gamma):
    G = 0.0
    for t, r in enumerate(rewards):
        G += (gamma ** t) * r
    return G


def theoretical_bound(M, gamma):
    return M / (1 - gamma)


gammas = np.linspace(0.5, 0.999, 300)
M = 1.0
bounds = theoretical_bound(M, gammas)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gammas, bounds, color='steelblue', linewidth=2)
axes[0].set_xlabel('Discount factor γ')
axes[0].set_ylabel('Return bound M / (1 - γ)')
axes[0].set_title('Infinite-Horizon Return Bound vs γ')
axes[0].axvline(0.99, color='orange', linestyle='--', label='γ=0.99')
axes[0].legend()

# Discount weight over time steps
steps = np.arange(0, 200)
for g, col in [(0.99, 'steelblue'), (0.95, 'orange'), (0.80, 'tomato')]:
    axes[1].plot(steps, g ** steps, label=f'γ={g}', color=col)
axes[1].set_xlabel('Time step t')
axes[1].set_ylabel('γᵗ (discount weight)')
axes[1].set_title('Discount Weight Decay Over Time')
axes[1].legend()

plt.tight_layout()
plt.show()

# Numerical demonstration
example_rewards = [1.0] * 100
for g in [0.99, 0.95, 0.80]:
    G = discounted_return(example_rewards, g)
    bound = theoretical_bound(1.0, g)
    print(f'γ={g:.2f}  |  G(100 steps) = {G:.3f}  |  Theoretical bound = {bound:.3f}')

---
## 3. Policy: Deterministic and Stochastic

Because of the Markov property, the optimal action at time $t$ only depends on the current state $s_t$ — not on history. This justifies defining a **policy** as a function of state alone.

### Deterministic policy

$$\pi: \mathcal{S} \to \mathcal{A}$$

Maps each state to a single action. There always exists an optimal deterministic policy — if two actions tie, pick either; randomizing cannot improve on the best.

**Proof sketch**: For any randomized policy $\pi$ mixing actions $a_1, a_2$ with probabilities $p, 1-p$, the expected payoff is a convex combination of the payoffs under $a_1$ and $a_2$. At least one of them achieves a payoff $\geq$ the convex combination. So the deterministic policy that always picks the better action is at least as good.

### Stochastic policy

$$\pi(a \mid s; \theta): \mathcal{S} \to \Delta(\mathcal{A})$$

Outputs a probability distribution over actions, parameterized by $\theta$. Even though a stochastic policy is suboptimal at convergence, it is necessary during training because:
- It provides **smooth gradients** — the probability of an action changes continuously as $\theta$ changes
- Deterministic policies produce hard switches between actions, which are not differentiable with respect to policy parameters

### Softmax parameterization

For discrete action spaces with $|\mathcal{A}| = K$, parameterize with preference logits $\theta \in \mathbb{R}^{|\mathcal{S}| \times K}$:

$$\pi(a \mid s; \theta) = \frac{\exp(\theta_{s,a})}{\sum_{a'} \exp(\theta_{s,a'})}$$

In [ ]:
def softmax(logits):
    logits = logits - logits.max()  # numerical stability
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum()


class StochasticPolicy:
    def __init__(self, n_states, n_actions):
        self.theta = np.zeros((n_states, n_actions))

    def action_probs(self, s):
        return softmax(self.theta[s])

    def sample_action(self, s):
        probs = self.action_probs(s)
        return np.random.choice(len(probs), p=probs)

    def log_prob(self, s, a):
        return np.log(self.action_probs(s)[a] + 1e-10)


class DeterministicPolicy:
    """Go right if s < goal, else go left (optimal for no obstacles)."""
    def __init__(self, goal):
        self.goal = goal

    def action(self, s):
        return 1 if s < self.goal else 0   # 1=right, 0=left


n_states, n_actions = mdp.n_states, len(mdp.actions)
stoch_policy = StochasticPolicy(n_states, n_actions)
det_policy   = DeterministicPolicy(mdp.goal)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stochastic policy probabilities (initialized uniform)
probs_matrix = np.array([stoch_policy.action_probs(s) for s in range(n_states)])
im = axes[0].imshow(probs_matrix.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
axes[0].set_xlabel('State s')
axes[0].set_ylabel('Action index')
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(mdp.actions)
axes[0].set_title('Stochastic Policy π(a|s;θ) — Initialized Uniform')
plt.colorbar(im, ax=axes[0], label='Probability')

# Deterministic policy
det_actions = [det_policy.action(s) for s in range(n_states)]
axes[1].bar(range(n_states), det_actions, color=['#e74c3c' if a == 0 else '#2ecc71' for a in det_actions])
axes[1].set_xlabel('State s')
axes[1].set_ylabel('Action (0=left, 1=right)')
axes[1].set_title('Deterministic Policy π*(s) — Go Toward Goal')
axes[1].axvline(mdp.goal - 0.5, color='black', linestyle='--', label='goal')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Stochastic policy at s=3: {dict(zip(mdp.actions, probs_matrix[3].round(3)))}')

---
## 4. Value Function

The **value function** under policy $\pi$ measures the expected discounted return starting from state $s$:

$$V^\pi(s) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t R(s_t) \,\Bigg|\, s_0 = s, \, a_t \sim \pi(\cdot \mid s_t)\right]$$

### Optimal value function

$$V^*(s) = \max_{\pi} V^\pi(s)$$

The optimal policy is:

$$\pi^*(s) = \arg\max_{\pi} V^\pi(s)$$

### Monte Carlo estimate

For a given policy $\pi$, roll out many trajectories starting from $s$ and average their returns:

$$\hat{V}^\pi(s) = \frac{1}{N} \sum_{n=1}^{N} G(\tau^{(n)}), \quad \tau^{(n)} \text{ starts from } s, \text{ actions from } \pi$$

By the law of large numbers, $\hat{V}^\pi(s) \to V^\pi(s)$ as $N \to \infty$.

In [ ]:
def rollout(mdp, policy_fn, s0, max_steps=200):
    """Run one trajectory. Returns discounted return."""
    s = s0
    total_return = 0.0
    for t in range(max_steps):
        a_idx = policy_fn(s)
        a = mdp.actions[a_idx]
        s_next, r = mdp.step(s, a)
        total_return += (mdp.gamma ** t) * r
        s = s_next
        if s == mdp.goal:
            break
    return total_return


def monte_carlo_value(mdp, policy_fn, n_rollouts=500):
    V = np.zeros(mdp.n_states)
    for s in range(mdp.n_states):
        returns = [rollout(mdp, policy_fn, s) for _ in range(n_rollouts)]
        V[s] = np.mean(returns)
    return V


# Compare value under deterministic (near-optimal) vs random policy
np.random.seed(42)
V_det    = monte_carlo_value(mdp, det_policy.action, n_rollouts=300)
V_random = monte_carlo_value(mdp, lambda s: np.random.randint(2), n_rollouts=300)

plt.figure(figsize=(10, 4))
plt.plot(range(mdp.n_states), V_det,    'o-', color='#2ecc71', label='Deterministic (near-optimal) policy')
plt.plot(range(mdp.n_states), V_random, 's-', color='#e74c3c', label='Random policy')
plt.xlabel('Starting state s')
plt.ylabel('$V^\\pi(s)$')
plt.title('Monte Carlo Estimate of $V^\\pi(s)$ — Deterministic vs Random Policy')
plt.axvline(mdp.goal, color='black', linestyle='--', alpha=0.5, label='goal')
plt.legend()
plt.show()

print('V_det  (s=0 to 9):', V_det.round(3))
print('V_rand (s=0 to 9):', V_random.round(3))

---
## 5. Bellman Equation

The key insight: the value function satisfies a **self-consistent recursion** that eliminates the infinite sum by turning it into a fixed-point equation.

### Derivation

Start from the definition:

$$V^\pi(s) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t R(s_t) \,\Bigg|\, s_0 = s\right]$$

**Step 1**: Peel off the first term ($t=0$ contributes $R(s)$ deterministically since $s_0 = s$):

$$V^\pi(s) = R(s) + \mathbb{E}\left[\sum_{t=1}^{\infty} \gamma^t R(s_t) \,\Bigg|\, s_0 = s\right]$$

**Step 2**: Factor out $\gamma$:

$$V^\pi(s) = R(s) + \gamma \, \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t R(s_{t+1}) \,\Bigg|\, s_0 = s\right]$$

**Step 3**: The inner expectation is $V^\pi(s_1)$ once we know $s_1$. Marginalize over $s_1$:

$$V^\pi(s) = R(s) + \gamma \sum_{s'} P_{s, \pi(s)}(s') \, V^\pi(s')$$

This is the **Bellman equation for policy $\pi$**. It is a system of $|\mathcal{S}|$ linear equations in $|\mathcal{S}|$ unknowns $\{V^\pi(s)\}$.

### Matrix form

Define:
- $\mathbf{v} = [V^\pi(s_0), \ldots, V^\pi(s_{n-1})]^\top \in \mathbb{R}^n$
- $\mathbf{r} = [R(s_0), \ldots, R(s_{n-1})]^\top \in \mathbb{R}^n$
- $T^\pi \in \mathbb{R}^{n \times n}$ where $T^\pi_{s,s'} = P_{s, \pi(s)}(s')$

Then:

$$\mathbf{v} = \mathbf{r} + \gamma T^\pi \mathbf{v} \quad \Rightarrow \quad (I - \gamma T^\pi) \mathbf{v} = \mathbf{r} \quad \Rightarrow \quad \mathbf{v} = (I - \gamma T^\pi)^{-1} \mathbf{r}$$

In [ ]:
def bellman_policy_evaluation(mdp, policy_fn):
    """
    Exact Bellman solution: V = (I - γ T^π)^{-1} r
    policy_fn: s -> action index
    """
    n = mdp.n_states
    # Build transition matrix T^π
    T_pi = np.zeros((n, n))
    for s in range(n):
        a_idx = policy_fn(s)
        a = mdp.actions[a_idx]
        T_pi[s] = mdp.P[s][a]

    A_mat = np.eye(n) - mdp.gamma * T_pi
    V = np.linalg.solve(A_mat, mdp.R)
    return V


V_exact_det    = bellman_policy_evaluation(mdp, det_policy.action)
V_exact_random = bellman_policy_evaluation(mdp, lambda s: 1)  # always go right

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(mdp.n_states), V_exact_det,    'o-', color='#2ecc71', label='Exact (Bellman)')
axes[0].plot(range(mdp.n_states), V_det,           's--', color='gray',   label='MC estimate')
axes[0].set_xlabel('State s')
axes[0].set_ylabel('$V^\\pi(s)$')
axes[0].set_title('Bellman Exact vs Monte Carlo — Deterministic Policy')
axes[0].legend()

# Show residual |V_exact - V_mc|
axes[1].bar(range(mdp.n_states), np.abs(V_exact_det - V_det), color='steelblue')
axes[1].set_xlabel('State s')
axes[1].set_ylabel('|V_exact - V_mc|')
axes[1].set_title('Monte Carlo Estimation Error per State')

plt.tight_layout()
plt.show()

print('Exact V (det policy):', V_exact_det.round(4))
print('Max MC error:', np.abs(V_exact_det - V_det).max().round(4))

---
## 6. Policy Gradient — The Log-Derivative Trick

The goal is to maximize the expected return over a parameterized stochastic policy $\pi_\theta$:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t R(s_t)\right]$$

### The challenge

$\theta$ does not appear inside the expectation — it only governs the **sampling distribution**. Naively moving $\nabla_\theta$ inside the expectation gives zero, because $R(s_t)$ does not depend on $\theta$.

### Log-derivative trick (REINFORCE)

Write the expectation as an integral:

$$J(\theta) = \int p_\theta(\tau) \, f(\tau) \, d\tau, \quad f(\tau) = \sum_t \gamma^t R(s_t)$$

Apply $\nabla_\theta$ inside the integral and use the identity $\nabla_\theta \log p = \nabla_\theta p / p$:

$$\nabla_\theta J(\theta) = \int \nabla_\theta p_\theta(\tau) \, f(\tau) \, d\tau = \int p_\theta(\tau) \, \nabla_\theta \log p_\theta(\tau) \, f(\tau) \, d\tau$$

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\nabla_\theta \log p_\theta(\tau) \cdot f(\tau)\right]}$$

### Simplifying $\log p_\theta(\tau)$

The trajectory probability factorizes:

$$p_\theta(\tau) = p(s_0) \prod_{t=0}^{T} \pi_\theta(a_t \mid s_t) \, P_{s_t, a_t}(s_{t+1})$$

Taking the log and differentiating with respect to $\theta$:

$$\nabla_\theta \log p_\theta(\tau) = \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t \mid s_t)$$

The terms $\log p(s_0)$ and $\log P_{s_t, a_t}(s_{t+1})$ vanish because they do not depend on $\theta$.

### Final REINFORCE gradient estimator

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\left(\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t \mid s_t)\right) \cdot \left(\sum_{t=0}^{T} \gamma^t R(s_t)\right)\right]$$

In practice, estimated using $N$ sampled trajectories:

$$\widehat{\nabla_\theta J}(\theta) = \frac{1}{N} \sum_{n=1}^{N} \left(\sum_{t} \nabla_\theta \log \pi_\theta(a_t^{(n)} \mid s_t^{(n)})\right) G^{(n)}$$

In [ ]:
def softmax_grad(logits, a_idx):
    """Gradient of log π(a|s;θ) with respect to θ_s (the logit vector for state s)."""
    probs = softmax(logits)
    # d/d θ_{s,a'} log π(a|s) = 1[a'=a] - π(a'|s)
    grad = -probs.copy()
    grad[a_idx] += 1.0
    return grad


def collect_trajectory(mdp, policy, max_steps=100):
    s0 = np.random.randint(mdp.n_states)
    s = s0
    trajectory = []
    for t in range(max_steps):
        a_idx = policy.sample_action(s)
        a = mdp.actions[a_idx]
        s_next, r = mdp.step(s, a)
        trajectory.append((s, a_idx, r))
        s = s_next
        if s == mdp.goal:
            break
    return trajectory


def compute_returns(trajectory, gamma):
    """Discounted return G_t for each step t."""
    T = len(trajectory)
    G = np.zeros(T)
    cumulative = 0.0
    for t in reversed(range(T)):
        cumulative = trajectory[t][2] + gamma * cumulative
        G[t] = cumulative
    return G


def reinforce_gradient(mdp, policy, trajectories):
    """Estimate ∇_θ J(θ) from a batch of trajectories."""
    grad_theta = np.zeros_like(policy.theta)
    for traj in trajectories:
        G_t = compute_returns(traj, mdp.gamma)
        total_G = G_t[0]  # full trajectory return from t=0
        for t, (s, a_idx, r) in enumerate(traj):
            grad_theta[s] += softmax_grad(policy.theta[s], a_idx) * total_G
    grad_theta /= len(trajectories)
    return grad_theta


# Demonstrate: what does the gradient look like at a single trajectory?
np.random.seed(42)
demo_policy = StochasticPolicy(mdp.n_states, len(mdp.actions))
demo_traj = collect_trajectory(mdp, demo_policy, max_steps=30)
G_t = compute_returns(demo_traj, mdp.gamma)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

states_visited = [step[0] for step in demo_traj]
actions_taken  = [step[1] for step in demo_traj]
rewards_got    = [step[2] for step in demo_traj]

axes[0].plot(states_visited, 'o-', color='steelblue', label='State s_t')
axes[0].axhline(mdp.goal, color='orange', linestyle='--', label=f'Goal (s={mdp.goal})')
axes[0].set_xlabel('Time step t')
axes[0].set_ylabel('State')
axes[0].set_title('Sample Trajectory (random policy)')
axes[0].legend()

axes[1].plot(G_t, 's-', color='#e74c3c', label='$G_t$ (discounted return from t)')
axes[1].set_xlabel('Time step t')
axes[1].set_ylabel('$G_t$')
axes[1].set_title('Discounted Return $G_t$ Along Trajectory')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Trajectory length: {len(demo_traj)}  |  Total return G_0: {G_t[0]:.4f}')

---
## 7. REINFORCE Training Loop

The REINFORCE algorithm:

1. Initialize policy parameters $\theta$
2. Repeat:
   a. Sample $N$ trajectories $\{\tau^{(n)}\}_{n=1}^N$ using current $\pi_\theta$
   b. Compute returns $G^{(n)}$ for each trajectory
   c. Estimate gradient: $\hat{g} = \frac{1}{N} \sum_n \sum_t \nabla_\theta \log \pi_\theta(a_t^{(n)} \mid s_t^{(n)}) \cdot G^{(n)}$
   d. Update: $\theta \leftarrow \theta + \alpha \hat{g}$ (gradient **ascent** — maximizing $J$)

### Why sampling is sufficient

We never enumerate all trajectories — we just sample $N$ of them and take an empirical mean. This is valid because the gradient estimator is **unbiased**:

$$\mathbb{E}_{\tau \sim \pi_\theta}\left[\hat{g}\right] = \nabla_\theta J(\theta)$$

The variance shrinks as $1/N$ with more trajectories. Variance reduction techniques (baselines, advantage functions) are commonly applied, but even the raw estimator converges.

In [ ]:
def reinforce_train(mdp, n_iterations=300, n_trajectories=20, lr=0.05, max_steps=100):
    np.random.seed(42)
    policy = StochasticPolicy(mdp.n_states, len(mdp.actions))
    history = []

    for iteration in range(n_iterations):
        trajectories = [collect_trajectory(mdp, policy, max_steps) for _ in range(n_trajectories)]

        # Mean return across trajectories (monitoring metric)
        mean_return = np.mean([compute_returns(traj, mdp.gamma)[0] for traj in trajectories])
        history.append(mean_return)

        # Gradient ascent step
        grad = reinforce_gradient(mdp, policy, trajectories)
        policy.theta += lr * grad

    return policy, history


trained_policy, return_history = reinforce_train(
    mdp, n_iterations=300, n_trajectories=20, lr=0.05, max_steps=100
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training curve
window = 20
smoothed = np.convolve(return_history, np.ones(window)/window, mode='valid')
axes[0].plot(return_history, alpha=0.3, color='steelblue', label='Raw return')
axes[0].plot(range(window-1, len(return_history)), smoothed, color='steelblue', linewidth=2, label=f'{window}-iter moving avg')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Mean discounted return')
axes[0].set_title('REINFORCE Training Curve')
axes[0].legend()

# Learned policy probabilities
learned_probs = np.array([trained_policy.action_probs(s) for s in range(mdp.n_states)])
im = axes[1].imshow(learned_probs.T, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_xlabel('State s')
axes[1].set_ylabel('Action')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(mdp.actions)
axes[1].set_title('Learned Policy π(a|s;θ) After REINFORCE')
plt.colorbar(im, ax=axes[1], label='Probability')

plt.tight_layout()
plt.show()

print('Learned action probs (prob of "right") per state:')
for s in range(mdp.n_states):
    p_right = learned_probs[s, 1]
    print(f'  s={s}: P(right)={p_right:.3f}  |  preferred={'right' if p_right > 0.5 else 'left'}')

---
## 8. Variance and the Baseline

The REINFORCE gradient estimator is unbiased but can have high variance, slowing convergence.

### Variance of the estimator

$$\text{Var}\left[\hat{g}\right] = \frac{1}{N} \text{Var}\left[\nabla_\theta \log \pi_\theta(a \mid s) \cdot G\right]$$

Since $G$ can be large in magnitude (positive or negative), the per-sample gradient estimate is noisy.

### Baseline subtraction

The key identity: for any function $b(s)$ that does not depend on the action,

$$\mathbb{E}_{a \sim \pi_\theta(\cdot \mid s)}\left[\nabla_\theta \log \pi_\theta(a \mid s) \cdot b(s)\right] = 0$$

**Proof**:
$$\mathbb{E}_a\left[\nabla_\theta \log \pi \cdot b\right] = b \int \pi_\theta(a \mid s) \nabla_\theta \log \pi_\theta(a \mid s) \, da = b \int \nabla_\theta \pi_\theta(a \mid s) \, da = b \, \nabla_\theta 1 = 0$$

Therefore, replacing $G$ with $(G - b)$ leaves the gradient **unbiased** but can **reduce variance**:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot (G_t - b)\right]$$

The optimal baseline $b = \mathbb{E}[G]$ (mean return) minimizes variance. The quantity $(G_t - b)$ is called the **advantage** — it measures how much better trajectory $\tau$ is compared to the average.

In [ ]:
def reinforce_with_baseline(mdp, n_iterations=300, n_trajectories=20, lr=0.05, max_steps=100):
    np.random.seed(42)
    policy = StochasticPolicy(mdp.n_states, len(mdp.actions))
    history = []
    grad_var_history = []

    for iteration in range(n_iterations):
        trajectories = [collect_trajectory(mdp, policy, max_steps) for _ in range(n_trajectories)]
        returns = [compute_returns(traj, mdp.gamma)[0] for traj in trajectories]
        mean_return = np.mean(returns)
        history.append(mean_return)

        # Baseline = mean return across batch
        baseline = mean_return

        grad_theta = np.zeros_like(policy.theta)
        per_traj_grads = []
        for traj, G in zip(trajectories, returns):
            advantage = G - baseline
            traj_grad = np.zeros_like(policy.theta)
            for s, a_idx, r in traj:
                traj_grad[s] += softmax_grad(policy.theta[s], a_idx) * advantage
            per_traj_grads.append(traj_grad)
            grad_theta += traj_grad
        grad_theta /= len(trajectories)

        grad_var = np.var([g.sum() for g in per_traj_grads])
        grad_var_history.append(grad_var)

        policy.theta += lr * grad_theta

    return policy, history, grad_var_history


_, history_no_baseline, _ = reinforce_train(mdp, n_iterations=300, n_trajectories=20, lr=0.05)
_, history_baseline, var_history = reinforce_with_baseline(mdp, n_iterations=300, n_trajectories=20, lr=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

w = 20
sm_nb = np.convolve(history_no_baseline, np.ones(w)/w, mode='valid')
sm_b  = np.convolve(history_baseline,    np.ones(w)/w, mode='valid')
x_sm  = range(w-1, 300)

axes[0].plot(x_sm, sm_nb, color='#e74c3c', linewidth=2, label='No baseline')
axes[0].plot(x_sm, sm_b,  color='#2ecc71', linewidth=2, label='With baseline')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Mean return (smoothed)')
axes[0].set_title('REINFORCE: Baseline Reduces Variance')
axes[0].legend()

sm_var = np.convolve(var_history, np.ones(w)/w, mode='valid')
axes[1].plot(x_sm, sm_var, color='steelblue', linewidth=2)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Gradient variance (smoothed)')
axes[1].set_title('Gradient Variance Over Training (With Baseline)')

plt.tight_layout()
plt.show()

print(f'Final mean return — no baseline: {np.mean(history_no_baseline[-50:]):.4f}')
print(f'Final mean return — baseline:    {np.mean(history_baseline[-50:]):.4f}')

---
## 9. Policy Convergence and Learned Behavior

At convergence, the stochastic policy $\pi_\theta$ should assign high probability to the action that moves toward the goal from each state. The learned $\theta$ parameters encode the preference logits, and the softmax turns them into probabilities.

### What convergence looks like

For the 1D grid world with goal at $s = 9$:

- For states $s < 9$: $\pi_\theta(\text{right} \mid s) \to 1$
- For states $s \geq 9$: policy is irrelevant (absorbing goal), but logits may show left preference

The trajectory quality should improve monotonically — shorter paths to goal, higher discounted return.

### Connection to LLMs

In language model alignment (RLHF, GRPO):
- States = token sequences generated so far
- Actions = next token to emit
- Policy = language model $\pi_\theta$
- Reward = scalar from a reward model (or rule-based verifier)
- REINFORCE gradient = same formula, applied at the token level

The only structural difference is the scale of the state/action spaces, not the algorithm.

In [ ]:
def evaluate_policy(mdp, policy, n_episodes=200, max_steps=100):
    returns, lengths = [], []
    for _ in range(n_episodes):
        s = np.random.randint(mdp.n_states)
        G, t = 0.0, 0
        for t in range(max_steps):
            a_idx = policy.sample_action(s)
            a = mdp.actions[a_idx]
            s, r = mdp.step(s, a)
            G += (mdp.gamma ** t) * r
            if s == mdp.goal:
                break
        returns.append(G)
        lengths.append(t + 1)
    return np.array(returns), np.array(lengths)


np.random.seed(0)
init_policy = StochasticPolicy(mdp.n_states, len(mdp.actions))  # uniform random
final_policy, _, _ = reinforce_with_baseline(mdp, n_iterations=400, n_trajectories=30, lr=0.05)

ret_init,  len_init  = evaluate_policy(mdp, init_policy,  n_episodes=300)
ret_final, len_final = evaluate_policy(mdp, final_policy, n_episodes=300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ret_init,  bins=30, alpha=0.6, color='#e74c3c', label='Before training')
axes[0].hist(ret_final, bins=30, alpha=0.6, color='#2ecc71', label='After training')
axes[0].axvline(ret_init.mean(),  color='#e74c3c', linestyle='--', linewidth=2)
axes[0].axvline(ret_final.mean(), color='#2ecc71', linestyle='--', linewidth=2)
axes[0].set_xlabel('Discounted return G')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Return Distribution: Before vs After REINFORCE')
axes[0].legend()

axes[1].hist(len_init,  bins=range(1, 60), alpha=0.6, color='#e74c3c', label='Before training')
axes[1].hist(len_final, bins=range(1, 60), alpha=0.6, color='#2ecc71', label='After training')
axes[1].set_xlabel('Trajectory length (steps)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Trajectory Length: Before vs After REINFORCE')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Mean return  — before: {ret_init.mean():.4f}  |  after: {ret_final.mean():.4f}')
print(f'Mean length  — before: {len_init.mean():.1f}  |  after: {len_final.mean():.1f}')
print(f'Return improvement: {(ret_final.mean() - ret_init.mean()):.4f}')